<a href="https://colab.research.google.com/github/hemachandranrcs24-spec/BIS_LAB/blob/main/week21bm24cs115_hemachandranr_particle_swarm_optimiazaiton_cyberattack_dataset_problem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import random
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

# 1. Define Realistic Cyber Network Features
feature_names = [
    "duration", "src_bytes", "dst_bytes", "land", "wrong_fragment", "urgent",
    "hot", "num_failed_logins", "logged_in", "num_compromised", "root_shell",
    "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate", "dst_host_same_src_port_rate"
]

# Generate simulated network traffic dataset (normal vs. attack)
X, y = make_classification(
    n_samples=2000,
    n_features=len(feature_names),
    n_informative=15,
    n_classes=2,
    random_state=42
)

# Split into training and test validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

NUM_FEATURES = len(feature_names)
PARAM_BITS = 6
CHROMOSOME_LENGTH = NUM_FEATURES + PARAM_BITS

# PSO Parameters
SWARM_SIZE = 20
ITERATIONS = 10
W = 0.5        # Inertia weight
C1 = 1.5       # Cognitive coefficient (personal best influence)
C2 = 1.5       # Social coefficient (global best influence)

def sigmoid(v):
    return 1 / (1 + np.exp(-np.clip(v, -10, 10)))

def decode_position(position):
    feature_mask = position[:NUM_FEATURES].astype(bool)
    param_bits = position[NUM_FEATURES:]

    # Decode 3 bits for n_estimators (Range: 10 to 150)
    n_est_bits = "".join(str(int(b)) for b in param_bits[:3])
    n_estimators = 10 + int(n_est_bits, 2) * 20

    # Decode 3 bits for max_depth (Range: 3 to 24)
    depth_bits = "".join(str(int(b)) for b in param_bits[3:])
    max_depth = 3 + int(depth_bits, 2) * 3

    return feature_mask, n_estimators, max_depth

def calculate_fitness(position):
    feature_mask, n_estimators, max_depth = decode_position(position)

    # If no features are selected, penalize with zero fitness
    if not any(feature_mask):
        return 0.0

    X_tr_selected = X_train[:, feature_mask]
    X_ts_selected = X_test[:, feature_mask]

    # Train detector on selected network features
    clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, n_jobs=-1)
    clf.fit(X_tr_selected, y_train)
    predictions = clf.predict(X_ts_selected)

    # Evaluate Intrusion Detection metrics
    f1 = f1_score(y_test, predictions, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()

    # False Alarm Rate
    far = fp / (fp + tn) if (fp + tn) > 0 else 1.0

    # Feature overhead ratio
    feature_ratio = np.sum(feature_mask) / NUM_FEATURES

    # Objective function weighting
    fitness = (0.6 * f1) + (0.2 * (1 - far)) + (0.2 * (1 - feature_ratio))
    return fitness

# Initialize Swarm Positions (continuous values) and Velocities
X_position = np.random.uniform(-3, 3, size=(SWARM_SIZE, CHROMOSOME_LENGTH))
V_velocity = np.random.uniform(-1, 1, size=(SWARM_SIZE, CHROMOSOME_LENGTH))

# Map to initial binary values using sigmoid thresholding
binary_positions = (sigmoid(X_position) > 0.5).astype(int)

# Track histories
pbest_position = np.copy(X_position)
pbest_binary = np.copy(binary_positions)
pbest_fitness = np.array([calculate_fitness(p) for p in binary_positions])

# Track global leader
gbest_idx = np.argmax(pbest_fitness)
gbest_fitness = pbest_fitness[gbest_idx]
gbest_position = np.copy(X_position[gbest_idx])
gbest_binary = np.copy(binary_positions[gbest_idx])

print(f"{'Iter':<6}{'Best Fitness':<15}{'Features Selected':<20}{'n_estimators':<15}{'max_depth'}")
print("-" * 65)

# Execute Particle Swarm Optimization Loop
for iteration in range(ITERATIONS):
    for i in range(SWARM_SIZE):
        r1 = np.random.rand(CHROMOSOME_LENGTH)
        r2 = np.random.rand(CHROMOSOME_LENGTH)

        # Calculate new velocity vector
        V_velocity[i] = (W * V_velocity[i]) + \
                        (C1 * r1 * (pbest_position[i] - X_position[i])) + \
                        (C2 * r2 * (gbest_position - X_position[i]))

        # Apply velocity vectors to shift coordinates
        X_position[i] = X_position[i] + V_velocity[i]

        # Constraint mapping: run values through continuous sigmoid mapping into discrete bits
        binary_positions[i] = (sigmoid(X_position[i]) > 0.5).astype(int)

        # Evaluate performance metrics at new site
        current_fit = calculate_fitness(binary_positions[i])

        # Update Personal Tracker
        if current_fit > pbest_fitness[i]:
            pbest_fitness[i] = current_fit
            pbest_position[i] = np.copy(X_position[i])
            pbest_binary[i] = np.copy(binary_positions[i])

        # Update Swarm Global Leader
        if current_fit > gbest_fitness:
            gbest_fitness = current_fit
            gbest_position = np.copy(X_position[i])
            gbest_binary = np.copy(binary_positions[i])

    feat_mask, n_est, depth = decode_position(gbest_binary)
    print(f"{iteration+1:<6}{gbest_fitness:<15.4f}{np.sum(feat_mask):<20}{n_est:<15}{depth}")

best_features, best_n_est, best_depth = decode_position(gbest_binary)
selected_feature_names = [feature_names[idx] for idx, val in enumerate(best_features) if val]

print("\n" + "="*50 + "\nCYBER ATTACK DETECTION OPTIMIZATION COMPLETE (PSO)\n" + "="*50)
print(f"Optimal Fitness Score: {gbest_fitness:.4f}")
print(f"Optimized Hyperparameters: n_estimators={best_n_est}, max_depth={best_depth}")
print(f"Selected Cyber Features ({len(selected_feature_names)}/{NUM_FEATURES}): {selected_feature_names}")


Iter  Best Fitness   Features Selected   n_estimators   max_depth
-----------------------------------------------------------------
1     0.8079         14                  30             15
2     0.8460         13                  30             15
3     0.8460         13                  30             15
4     0.8460         13                  30             15
5     0.8460         13                  30             15
6     0.8460         13                  30             15
7     0.8460         13                  30             15
8     0.8460         13                  30             15
9     0.8460         13                  30             15
10    0.8460         13                  30             15

CYBER ATTACK DETECTION OPTIMIZATION COMPLETE (PSO)
Optimal Fitness Score: 0.8460
Optimized Hyperparameters: n_estimators=30, max_depth=15
Selected Cyber Features (13/30): ['wrong_fragment', 'urgent', 'logged_in', 'num_compromised', 'root_shell', 'num_shells', 'serror_rate', 'r